In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv('../data/crime_dataset_india.csv')

In [3]:
assert df["Date of Occurrence"].notna().all()
df.sample(5)

,Report Number,Date Reported,Date of Occurrence,Time of Occurrence,City,Crime Code,Crime Description,Victim Age,Victim Gender,Weapon Used,Crime Domain,Police Deployed,Case Closed,Date Case Closed
741,742,02-02-2020 22:00,01-31-2020 21:00,01-02-2020 14:51,Ahmedabad,502,DOMESTIC VIOLENCE,62,M,Poison,Violent Crime,3,Yes,21-03-2020 22:00
10850,10851,29-03-2021 12:00,03-28-2021 02:00,28-03-2021 06:01,Kolkata,152,SHOPLIFTING,66,F,NaN,Other Crime,16,No,NaN
20449,20450,03-05-2022 21:00,05-02-2022 01:00,02-05-2022 21:49,Delhi,214,FRAUD,15,X,NaN,Other Crime,17,Yes,12-07-2022 21:00
820,821,06-02-2020 00:00,02-04-2020 04:00,05-02-2020 01:12,Hyderabad,408,FIREARM OFFENSE,76,F,Blunt Object,Fire Accident,5,Yes,11-03-2020 00:00
19834,19835,07-04-2022 01:00,04-06-2022 10:00,07-04-2022 08:30,Delhi,268,DOMESTIC VIOLENCE,25,F,Poison,Violent Crime,10,Yes,14-04-2022 01:00


In [4]:
# converting to date time format
df["Date of Occurrence"] = pd.to_datetime(df["Date of Occurrence"], format="%m-%d-%Y %H:%M")
df["Date Reported"]      = pd.to_datetime(df["Date Reported"],      format="%d-%m-%Y %H:%M")
df["Time of Occurrence"] = pd.to_datetime(df["Time of Occurrence"], format="%d-%m-%Y %H:%M")

In [5]:
df.head()

,Report Number,Date Reported,Date of Occurrence,Time of Occurrence,City,Crime Code,Crime Description,Victim Age,Victim Gender,Weapon Used,Crime Domain,Police Deployed,Case Closed,Date Case Closed
0,1,2020-01-02 00:00:00,2020-01-01 00:00:00,2020-01-01 01:11:00,Ahmedabad,576,IDENTITY THEFT,16,M,Blunt Object,Violent Crime,13,No,NaN
1,2,2020-01-01 19:00:00,2020-01-01 01:00:00,2020-01-01 06:26:00,Chennai,128,HOMICIDE,37,M,Poison,Other Crime,9,No,NaN
2,3,2020-01-02 05:00:00,2020-01-01 02:00:00,2020-01-01 14:30:00,Ludhiana,271,KIDNAPPING,48,F,Blunt Object,Other Crime,15,No,NaN
3,4,2020-01-01 05:00:00,2020-01-01 03:00:00,2020-01-01 14:46:00,Pune,170,BURGLARY,49,F,Firearm,Other Crime,1,Yes,29-04-2020 05:00
4,5,2020-01-01 21:00:00,2020-01-01 04:00:00,2020-01-01 16:51:00,Pune,421,VANDALISM,30,F,Other,Other Crime,18,Yes,08-01-2020 21:00


In [6]:
# # geocoding
# cities = df['City'].unique()
# city_coords = pd.read_csv('../data/IndiaCitiesLatLng.csv')[['city','lat','lng']]
# df = df.merge(city_coords, left_on="City",right_on='city',how='left') #left joining both cols


In [7]:
# standardizing city names
df["City"] = df['City'].str.strip().str.title()
print(df["City"].value_counts())

City
Delhi            5400
Mumbai           4415
Bangalore        3588
Hyderabad        2881
Kolkata          2518
Chennai          2493
Pune             2212
Ahmedabad        1817
Jaipur           1479
Lucknow          1456
Kanpur           1112
Surat            1111
Nagpur           1053
Agra              764
Ludhiana          761
Visakhapatnam     728
Thane             706
Ghaziabad         704
Indore            699
Patna             695
Bhopal            690
Meerut            395
Srinagar          371
Nashik            366
Vasai             362
Varanasi          355
Kalyan            355
Faridabad         354
Rajkot            320
Name: count, dtype: int64


In [8]:
world_cities = pd.read_csv('../data/worldcities.csv')[['city','lat','lng','country',"city_ascii"]]

In [9]:
def geocode_quality_report(df):

    if "lat" not in df.columns:
        raise ValueError(
            "Latitude column missing. Merge probably failed."
        )

    total=len(df)

    missing=df["lat"].isna().sum()

    pct=missing/total*100

    by_city=df.loc[df["lat"].isna(),"City"].value_counts()

    print(f"Coverage : {100-pct:.2f}%")

    return by_city

In [10]:
india_cities = world_cities[world_cities["country"] == "India"][["city_ascii", "lat", "lng"]].copy()
india_cities = india_cities.rename(columns={"city_ascii": "city"})
india_cities["city"] = india_cities["city"].str.strip().str.title()
india_cities = india_cities.drop_duplicates(subset="city", keep="first")

In [11]:
df.head()

,Report Number,Date Reported,Date of Occurrence,Time of Occurrence,City,Crime Code,Crime Description,Victim Age,Victim Gender,Weapon Used,Crime Domain,Police Deployed,Case Closed,Date Case Closed
0,1,2020-01-02 00:00:00,2020-01-01 00:00:00,2020-01-01 01:11:00,Ahmedabad,576,IDENTITY THEFT,16,M,Blunt Object,Violent Crime,13,No,NaN
1,2,2020-01-01 19:00:00,2020-01-01 01:00:00,2020-01-01 06:26:00,Chennai,128,HOMICIDE,37,M,Poison,Other Crime,9,No,NaN
2,3,2020-01-02 05:00:00,2020-01-01 02:00:00,2020-01-01 14:30:00,Ludhiana,271,KIDNAPPING,48,F,Blunt Object,Other Crime,15,No,NaN
3,4,2020-01-01 05:00:00,2020-01-01 03:00:00,2020-01-01 14:46:00,Pune,170,BURGLARY,49,F,Firearm,Other Crime,1,Yes,29-04-2020 05:00
4,5,2020-01-01 21:00:00,2020-01-01 04:00:00,2020-01-01 16:51:00,Pune,421,VANDALISM,30,F,Other,Other Crime,18,Yes,08-01-2020 21:00


In [12]:
from rapidfuzz import process

remaining = still_missing.index.tolist()
lookup_names = india_cities["city"].tolist()

for city in remaining:
    match, score, _ = process.extractOne(city, lookup_names)
    print(f"{city:20s} -> {match:20s} (score: {score:.0f})")

NameError: name 'still_missing' is not defined

In [ ]:
match, score, _ = process.extractOne("Visakhapatnam", india_cities["city"].tolist())
print(repr(match))  # repr shows hidden whitespace/characters that str hides

'Vishakhapatnam'


In [ ]:
rename_map = {
    "Nashik": "Nasik",
    "Vasai": "Vasai-Virar",
    "Visakhapatnam":"Vishakhapatnam"
}

In [ ]:
print(df.columns.tolist())

['Report Number', 'Date Reported', 'Date of Occurrence', 'Time of Occurrence', 'City', 'Crime Code', 'Crime Description', 'Victim Age', 'Victim Gender', 'Weapon Used', 'Crime Domain', 'Police Deployed', 'Case Closed', 'Date Case Closed', 'city_x', 'lat_x', 'lng_x', 'city_y', 'lat_y', 'lng_y']


In [ ]:
cols_to_drop = [
    "city","lat","lng",
    "city_x","lat_x","lng_x",
    "city_y","lat_y","lng_y"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [ ]:
duplicate_cols = [c for c in df.columns if c.endswith("_duplicate")]

df.drop(columns=duplicate_cols, inplace=True)

In [ ]:
df["City"] = df["City"].replace(rename_map)

df = df.merge(
    india_cities,
    left_on="City",
    right_on="city",
    how="left",
    suffixes=("", "_duplicate")
)
assert len(df)>0
assert "lat" in df.columns, f"Merge produced unexpected columns: {df.columns.tolist()}"

unmatched = geocode_quality_report(df)

Geocode coverage: 100.0% (40160/40160 rows)
Unmatched cities:
Series([], Name: count, dtype: int64)


In [ ]:
df.head()

,Report Number,Date Reported,Date of Occurrence,Time of Occurrence,City,Crime Code,Crime Description,Victim Age,Victim Gender,Weapon Used,Crime Domain,Police Deployed,Case Closed,Date Case Closed
0,1,2020-01-02 00:00:00,2020-01-01 00:00:00,2020-01-01 01:11:00,Ahmedabad,576,IDENTITY THEFT,16,M,Blunt Object,Violent Crime,13,No,NaN
1,2,2020-01-01 19:00:00,2020-01-01 01:00:00,2020-01-01 06:26:00,Chennai,128,HOMICIDE,37,M,Poison,Other Crime,9,No,NaN
2,3,2020-01-02 05:00:00,2020-01-01 02:00:00,2020-01-01 14:30:00,Ludhiana,271,KIDNAPPING,48,F,Blunt Object,Other Crime,15,No,NaN
3,4,2020-01-01 05:00:00,2020-01-01 03:00:00,2020-01-01 14:46:00,Pune,170,BURGLARY,49,F,Firearm,Other Crime,1,Yes,29-04-2020 05:00
4,5,2020-01-01 21:00:00,2020-01-01 04:00:00,2020-01-01 16:51:00,Pune,421,VANDALISM,30,F,Other,Other Crime,18,Yes,08-01-2020 21:00


In [13]:
plt.figure(figsize=(8,8))
plt.scatter(df["lon_j"], df["lat_j"], c=df["cluster"], cmap="tab20", s=3, alpha=0.5)
plt.title(f"DBSCAN clusters (eps={eps_km}km, minPts={min_pts})")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.show()
print(f"Found {hotspots.shape[0]} hotspot clusters, {(df['cluster']==-1).sum()} noise points")

KeyError: 'lon_j'

<Figure size 800x800 with 0 Axes>

In [ ]:
COVERAGE_THRESHOLD = 0.98  # fail if more than 2% of rows can't be geocoded

coverage = 1 - df["lat"].isna().sum() / len(df)
if coverage < COVERAGE_THRESHOLD:
    raise ValueError(f"Geocode coverage {coverage:.1%} below threshold — fix aliases before proceeding")

ValueError: Geocode coverage 96.4% below threshold — fix aliases before proceeding

In [ ]:
# adding random offset to these lat long points -> so they dont show up at very same space -> adding jitter
city_tier = {"Mumbai": 15, "Delhi": 15, "Bangalore": 12, "Chennai": 12}
default_spread = 6

def jitter(row):
    spread = city_tier.get(row["City"],default_spread) * 0.009
    return row['lat']+ np.random.normal(0,spread),row['lng'] + np.random.normal(0,spread)

df['lat_j'] , df['lon_j'] = zip(*df.apply(jitter,axis=1))

In [ ]:
print(df[["lat_j", "lon_j"]].isna().sum())

In [ ]:
missing_cities = df[df["lat_j"].isna()]["City"].value_counts()
print(missing_cities)

In [ ]:
lookup_cities = set(world_cities["city"].str.strip().str.title())
your_cities = set(df["City"].unique())

print("In your data but NOT in lookup:", your_cities - lookup_cities)
print("In lookup but not matched (sanity check):", len(lookup_cities))

In [ ]:
from sklearn.neighbors import NearestNeighbors
coords = np.radians(df[['lat_j','lon_j']].values)
min_pts = 15 
neighbors = NearestNeighbors(n_neighbors= min_pts, metric="haversine", algorithm="ball_tree").fit(coords)
distances, _ = neighbors.kneighbors(coords)
k_dist = np.sort(distances[:,-1])* 6371 #converting radians -> kms

In [ ]:
plt.plot(k_dist)
plt.ylabel('dist to 15th nearest neighbor (km)')

In [ ]:
from sklearn.cluster import DBSCAN
eps_km = 4.2
db = DBSCAN(eps=eps_km/6371, min_samples=min_pts, metric="haversine", algorithm="ball_tree").fit(coords)
df["cluster"] = db.labels_  # -1 = noise

In [ ]:
hotspots = df[df["cluster"] != -1].groupby("cluster").agg(
    incidents=("Report Number", "count"),
    violent_share=("Crime Domain", lambda x: (x == "Violent Crime").mean()),
    lat=("lat_j", "mean"),
    lon=("lon_j", "mean"),
)
hotspots["weight"] = hotspots["incidents"] * (1 + hotspots["violent_share"])
hotspots = hotspots.sort_values("weight", ascending=False)

# greedy allocation
n_units = 5
hotspots["units_assigned"] = 0
hotspots.iloc[:n_units, hotspots.columns.get_loc("units_assigned")] = 1
